#Ingesta de carpeta con archivos JSON

In [0]:
#1. Leer archivos JSON usando DataFrameReader de Spark

#Importamos las librerias que se van a utilizar
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from pyspark.sql.functions import col, concat, current_timestamp, lit

# Define el la estructura personName
production_country_schema = StructType(fields = [
    StructField("movieId", IntegerType(), True),
    StructField("countryId", IntegerType(), True)
])

# Cargamos el archivo utilizando la estructura definida
production_country_df = spark.read\
    .schema(production_country_schema)\
    .option("multiLine", "true")\
    .json("abfss://bronze@lsdata01.dfs.core.windows.net/production_country")

# Mostramos el resultado
display(production_country_df.filter(col("movieId") == 5))


In [0]:
#Paso 2 - Renombrar, añadir y dar formato a las columnas requeridas

production_country_renamed_df = production_country_df\
    .withColumnRenamed("movieId", "movie_id")\
    .withColumnRenamed("countryId", "country_id")\
    .withColumn("ingestion_date", current_timestamp())\
    .withColumn("enviroment", lit("Produccion"))
    
display(production_country_renamed_df.limit(10))


In [0]:
#Paso 4 - Guardar datos en datalake en formato parket 
production_country_renamed_df.write.mode("overwrite").parquet("abfss://silver@lsdata01.dfs.core.windows.net/production_countries")
df = spark.read.parquet("abfss://silver@lsdata01.dfs.core.windows.net/production_countries")
display(df)


In [0]:
%fs
ls abfss://silver@lsdata01.dfs.core.windows.net/production_countries